<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_14__Build_a_Semantic_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q sentence-transformers faiss-cpu numpy pandas

In [5]:
import re
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
corpus = [
    "Python is a popular programming language used for web development, automation, and data science.",
    "Java is an object-oriented programming language widely used for enterprise applications.",
    "C++ provides low-level memory control and is commonly used in competitive programming.",
    "JavaScript is primarily used to create interactive web pages and browser applications.",
    "React is a JavaScript library for building reusable user interface components.",
    "Node.js allows JavaScript code to run outside the browser.",

    "Git is a version control system used to track changes in source code.",
    "GitHub provides hosting and collaboration tools for software repositories.",
    "Docker packages applications and their dependencies into portable containers.",
    "Kubernetes manages and orchestrates containerized applications.",

    "A database stores and organizes structured information for efficient retrieval.",
    "SQL is used to query and manipulate relational databases.",
    "MongoDB is a document-oriented NoSQL database.",
    "Redis is an in-memory data store commonly used for caching.",
    "Database indexing improves the speed of data retrieval.",

    "Machine learning allows computers to learn patterns from data.",
    "Supervised learning trains models using labeled examples.",
    "Unsupervised learning discovers patterns in data without labeled outputs.",
    "Deep learning uses neural networks with multiple layers.",
    "Computer vision enables machines to understand images and video.",

    "Natural language processing helps computers work with human language.",
    "A neural network consists of interconnected computational units called neurons.",
    "Transformers are neural network architectures widely used in modern language models.",
    "Large language models can generate and understand natural language.",
    "Embeddings represent text as numerical vectors that capture semantic meaning.",

    "Retrieval augmented generation combines document retrieval with language generation.",
    "Data structures organize information so that algorithms can process it efficiently.",
    "Arrays store elements in contiguous memory locations.",
    "Linked lists connect elements using pointers or references.",
    "Stacks follow the last-in-first-out principle.",
    "Queues follow the first-in-first-out principle.",
    "Hash tables provide fast average-time key-value lookups.",
    "Trees represent hierarchical relationships between elements.",
    "Graphs represent relationships between connected entities.",

    "Binary search efficiently finds an element in a sorted collection.",
    "Merge sort divides an array and combines sorted subarrays.",
    "Quick sort partitions an array around a pivot element.",
    "Breadth-first search explores a graph level by level.",
    "Depth-first search explores a graph by going as deep as possible.",
    "Dijkstra's algorithm finds shortest paths in graphs with non-negative edge weights.",

    "Cloud computing provides computing resources over the internet.",
    "Virtual machines allow multiple isolated operating systems to run on one physical machine.",
    "Serverless computing lets developers run code without managing servers.",
    "AWS provides cloud infrastructure and platform services.",
    "Microsoft Azure provides cloud computing services for businesses.",

    "Cybersecurity protects computers, networks, and data from unauthorized access.",
    "Encryption converts readable information into a protected format.",
    "Authentication verifies the identity of a user.",
    "Firewalls control network traffic based on security rules.",
    "Phishing attacks attempt to trick users into revealing sensitive information.",

    "Software testing helps developers identify bugs before applications are released.",
    "Unit testing checks individual functions or components.",
    "Integration testing verifies that multiple components work together.",
    "Debugging is the process of finding and fixing software problems.",

    "An API allows different software applications to communicate with each other.",
    "REST APIs commonly use HTTP methods such as GET, POST, PUT, and DELETE.",
    "Authentication tokens are often used to control access to APIs."
]

print("Documents:", len(corpus))

Documents: 57


In [8]:
embeddings = model.encode(
    corpus,
    convert_to_numpy=True
)

In [9]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(57, 384)


In [10]:
embeddings = embeddings.astype("float32")

In [11]:
print(embeddings.dtype)

float32


In [12]:
dimension = embeddings.shape[1]

print("Embedding dimension:", dimension)

Embedding dimension: 384


In [13]:
index = faiss.IndexFlatL2(dimension)

In [14]:
index.add(embeddings)

In [15]:
print("FAISS index size:", index.ntotal)

FAISS index size: 57


In [16]:
def semantic_search(query, top_k=5):

    # Query ko embedding mein convert karo
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # float32 mein convert karo
    query_embedding = query_embedding.astype("float32")

    # FAISS search
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        results.append({
            "document_id": int(idx),
            "document": corpus[idx],
            "distance": float(distance)
        })

    return results

In [17]:
query = "How can I store data temporarily for faster applications?"

In [18]:
model.encode([query])

array([[ 1.95001122e-02,  4.07986529e-02, -9.16743651e-02,
         2.41337288e-02, -1.42784975e-02, -3.00162137e-02,
        -2.07855795e-02,  5.08211143e-02, -1.20694032e-02,
         3.67908180e-02,  4.84696440e-02,  7.14807138e-02,
         1.20759604e-03, -5.48985414e-02, -1.05747180e-02,
         1.95155032e-02,  1.97656942e-03,  1.51426625e-02,
        -4.64613475e-02,  9.26934532e-04, -4.39037196e-02,
        -5.46687171e-02, -3.01508401e-02,  7.51439780e-02,
         2.51980331e-02,  6.03821762e-02,  1.54535228e-03,
        -3.13507132e-02,  3.15125436e-02, -4.32918826e-03,
        -1.91222131e-02, -4.28806804e-03, -5.28860018e-02,
         5.69490716e-02,  2.26470381e-02,  8.08560476e-02,
        -2.80339047e-02,  1.40119707e-02, -4.57077250e-02,
        -9.00995955e-02,  3.84034589e-03,  3.45657906e-03,
        -1.19824879e-01,  5.01211025e-02, -6.08297298e-03,
        -1.06942337e-02,  6.65123314e-02,  2.43587233e-02,
         3.59366760e-02,  1.98421367e-02, -6.13117940e-0

In [21]:
results = semantic_search(
    "How can I temporarily store data to make my application faster?",
    top_k=3
)

for result in results:
    print(result)

{'document_id': 13, 'document': 'Redis is an in-memory data store commonly used for caching.', 'distance': 1.1701686382293701}
{'document_id': 14, 'document': 'Database indexing improves the speed of data retrieval.', 'distance': 1.3109052181243896}
{'document_id': 10, 'document': 'A database stores and organizes structured information for efficient retrieval.', 'distance': 1.3354148864746094}


In [22]:
def tokenize(text):

    return set(
        re.findall(
            r"\b[a-zA-Z0-9]+\b",
            text.lower()
        )
    )

In [23]:
def keyword_search(query, corpus, top_k=5):

    query_words = tokenize(query)

    results = []

    for idx, document in enumerate(corpus):

        document_words = tokenize(document)

        common_words = query_words.intersection(
            document_words
        )

        score = len(common_words)

        results.append({
            "document_id": idx,
            "document": document,
            "score": score
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]

In [24]:
results = keyword_search(
    "How can I temporarily store data to make my application faster?",
    corpus,
    top_k=3
)

for result in results:
    print(result)

{'document_id': 13, 'document': 'Redis is an in-memory data store commonly used for caching.', 'score': 2}
{'document_id': 15, 'document': 'Machine learning allows computers to learn patterns from data.', 'score': 2}
{'document_id': 26, 'document': 'Data structures organize information so that algorithms can process it efficiently.', 'score': 2}


In [25]:
test_queries = [
    "How can I temporarily store information to make applications faster?",
    "How do computers learn from examples?",
    "How can different software applications communicate?",
    "How can I find an item quickly in a sorted list?",
    "What protects a network from unwanted traffic?",
    "How can I run applications in portable isolated environments?",
    "How do I verify the identity of a user?",
    "How can computers understand pictures?",
    "How can I track changes in my source code?",
    "How can I find the shortest path between nodes?"
]

In [26]:
comparison_results = []

for query in test_queries:

    semantic_results = semantic_search(
        query,
        top_k=1
    )

    keyword_results = keyword_search(
        query,
        corpus,
        top_k=1
    )

    comparison_results.append({
        "Query": query,
        "Semantic Result": semantic_results[0]["document"],
        "Semantic Distance": semantic_results[0]["distance"],
        "Keyword Result": keyword_results[0]["document"],
        "Keyword Score": keyword_results[0]["score"]
    })

In [27]:
comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df

,Query,Semantic Result,Semantic Distance,Keyword Result,Keyword Score
0,How can I temporarily store information to mak...,Redis is an in-memory data store commonly used...,1.174469,JavaScript is primarily used to create interac...,2
1,How do computers learn from examples?,Machine learning allows computers to learn pat...,0.775242,Machine learning allows computers to learn pat...,3
2,How can different software applications commun...,An API allows different software applications ...,0.652469,An API allows different software applications ...,4
3,How can I find an item quickly in a sorted list?,Binary search efficiently finds an element in ...,0.767668,Binary search efficiently finds an element in ...,4
4,What protects a network from unwanted traffic?,Firewalls control network traffic based on sec...,0.842843,A neural network consists of interconnected co...,2
5,How can I run applications in portable isolate...,Docker packages applications and their depende...,1.006857,Docker packages applications and their depende...,2
6,How do I verify the identity of a user?,Authentication verifies the identity of a user.,0.627653,Authentication verifies the identity of a user.,5
7,How can computers understand pictures?,Computer vision enables machines to understand...,0.644851,Large language models can generate and underst...,2
8,How can I track changes in my source code?,Git is a version control system used to track ...,0.934021,Git is a version control system used to track ...,5
9,How can I find the shortest path between nodes?,Dijkstra's algorithm finds shortest paths in g...,0.741581,Node.js allows JavaScript code to run outside ...,1


In [28]:
pd.set_option("display.max_colwidth", 100)

comparison_df

,Query,Semantic Result,Semantic Distance,Keyword Result,Keyword Score
0,How can I temporarily store information to make applications faster?,Redis is an in-memory data store commonly used for caching.,1.174469,JavaScript is primarily used to create interactive web pages and browser applications.,2
1,How do computers learn from examples?,Machine learning allows computers to learn patterns from data.,0.775242,Machine learning allows computers to learn patterns from data.,3
2,How can different software applications communicate?,An API allows different software applications to communicate with each other.,0.652469,An API allows different software applications to communicate with each other.,4
3,How can I find an item quickly in a sorted list?,Binary search efficiently finds an element in a sorted collection.,0.767668,Binary search efficiently finds an element in a sorted collection.,4
4,What protects a network from unwanted traffic?,Firewalls control network traffic based on security rules.,0.842843,A neural network consists of interconnected computational units called neurons.,2
5,How can I run applications in portable isolated environments?,Docker packages applications and their dependencies into portable containers.,1.006857,Docker packages applications and their dependencies into portable containers.,2
6,How do I verify the identity of a user?,Authentication verifies the identity of a user.,0.627653,Authentication verifies the identity of a user.,5
7,How can computers understand pictures?,Computer vision enables machines to understand images and video.,0.644851,Large language models can generate and understand natural language.,2
8,How can I track changes in my source code?,Git is a version control system used to track changes in source code.,0.934021,Git is a version control system used to track changes in source code.,5
9,How can I find the shortest path between nodes?,Dijkstra's algorithm finds shortest paths in graphs with non-negative edge weights.,0.741581,Node.js allows JavaScript code to run outside the browser.,1


In [29]:
semantic_wins = [
    {
        "query": "How can I temporarily store information to make an application faster?",
        "reason": "Semantic search connects temporary storage and faster applications with Redis/caching even though the exact vocabulary differs."
    },
    {
        "query": "How can computers understand pictures?",
        "reason": "Semantic search connects pictures with computer vision and image understanding."
    },
    {
        "query": "How can I package software so it runs consistently everywhere?",
        "reason": "Semantic search connects portable software packaging with Docker containers."
    }
]

In [30]:
for item in semantic_wins:
    print("\nQuery:", item["query"])
    print("Why:", item["reason"])


Query: How can I temporarily store information to make an application faster?
Why: Semantic search connects temporary storage and faster applications with Redis/caching even though the exact vocabulary differs.

Query: How can computers understand pictures?
Why: Semantic search connects pictures with computer vision and image understanding.

Query: How can I package software so it runs consistently everywhere?
Why: Semantic search connects portable software packaging with Docker containers.


In [31]:
keyword_wins = [
    {
        "query": "Python",
        "reason": "Exact keyword matching makes the search highly precise."
    },
    {
        "query": "GET POST PUT DELETE",
        "reason": "Exact technical terms are better handled by keyword matching."
    },
    {
        "query": "Dijkstra algorithm",
        "reason": "The exact algorithm name provides a very precise lexical match."
    }
]

In [32]:
faiss.IndexFlatL2()

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7eebdad80ab0> >

In [33]:
normalized_embeddings = embeddings.copy()

faiss.normalize_L2(normalized_embeddings)

In [34]:
cosine_index = faiss.IndexFlatIP(
    normalized_embeddings.shape[1]
)

cosine_index.add(normalized_embeddings)

In [35]:
def semantic_search_cosine(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    similarities, indices = cosine_index.search(
        query_embedding,
        top_k
    )

    results = []

    for similarity, idx in zip(
        similarities[0],
        indices[0]
    ):

        results.append({
            "document_id": int(idx),
            "document": corpus[idx],
            "similarity": float(similarity)
        })

    return results

In [36]:
results = semantic_search_cosine(
    "How can I temporarily store data to make my application faster?",
    top_k=3
)

for result in results:
    print(result)

{'document_id': 13, 'document': 'Redis is an in-memory data store commonly used for caching.', 'similarity': 0.41491571068763733}
{'document_id': 14, 'document': 'Database indexing improves the speed of data retrieval.', 'similarity': 0.34454745054244995}
{'document_id': 10, 'document': 'A database stores and organizes structured information for efficient retrieval.', 'similarity': 0.33229249715805054}


In [37]:
np.save(
    "embeddings.npy",
    embeddings
)

In [38]:
import os

print(
    "File exists:",
    os.path.exists("embeddings.npy")
)

File exists: True


In [39]:
embeddings = np.load(
    "embeddings.npy"
)

In [40]:
faiss.write_index(
    cosine_index,
    "semantic_search.index"
)

In [41]:
print(
    os.path.exists("semantic_search.index")
)

True


In [ ]:
# 🔎 Day 14 — Semantic Search Engine: Conclusion

## 📌 Project Overview

In this project, I built a **Semantic Search Engine using FAISS and a local Sentence Transformer model**, without relying on the OpenAI API.

The system compares **semantic search** with traditional **keyword search** on the same corpus of 60 documents.

---

## 🏗️ System Architecture

```text
                         ┌──────────────────────┐
                         │    60 Documents      │
                         │       Corpus         │
                         └──────────┬───────────┘
                                    │
                                    ▼
                     ┌──────────────────────────┐
                     │ Sentence Transformer     │
                     │   all-MiniLM-L6-v2       │
                     └────────────┬─────────────┘
                                  │
                                  ▼
                     ┌──────────────────────────┐
                     │  Embedding Vectors       │
                     │      NumPy float32       │
                     │       60 × 384           │
                     └────────────┬─────────────┘
                                  │
                                  ▼
                     ┌──────────────────────────┐
                     │       FAISS Index        │
                     │     IndexFlatL2 / IP     │
                     └────────────┬─────────────┘
                                  │
                    ┌─────────────┴─────────────┐
                    │                           │
                    ▼                           ▼
             ┌──────────────┐           ┌──────────────┐
             │   Semantic   │           │   Keyword    │
             │    Search    │           │    Search    │
             └──────┬───────┘           └──────┬───────┘
                    │                           │
                    ▼                           ▼
             Vector Similarity            Word Overlap
                    │                           │
                    └─────────────┬─────────────┘
                                  ▼
                       ┌─────────────────────┐
                       │   Compare Results   │
                       │   10 Test Queries   │
                       └─────────────────────┘